# Model Optimization Tutorial

This tutorial describe the process of optimizing the user's model. The input to this tutorial is a HAR file in Hailo Model state (before optimization; with native weights) and the output will be a quantized HAR file with quantized weights.

Note: For full information about Optimization and Quantization, refer to the `Dataflow Compiler user guide / Model optimization` section.

**Requirements:**

* Run this code in Jupyter notebook. See the Introduction tutorial for more details.
* The user should review the complete Parsing Tutorial (or created the HAR file in other way)

**Recommendation:**

* To obtain best performance run this code with a GPU machine. For full information see the `Dataflow Compiler user guide / Model optimization` section.

**Contents:**

* Quick optimization tutorial
* In-depth optimization & evaluation tutorial
* Advanced Model Modifications tutorial
* Compression and Optimization levels

In [33]:
# General imports used throughout the tutorial
# file operations
import json
import os

import numpy as np
import tensorflow as tf
from IPython.display import SVG
from matplotlib import patches
from matplotlib import pyplot as plt
from PIL import Image
from tensorflow.python.eager.context import eager_mode

# import the hailo sdk client relevant classes
from hailo_sdk_client import ClientRunner, InferenceContext

%matplotlib inline

IMAGES_TO_VISUALIZE = 5

## Quick Optimization Tutorial

After the HAR file has been created (using either `runner.translate_tf_model` or `runner.translate_onnx_model`), the next step is to go through the optimization process.

The basic optimization is performed just by calling `runner.optimize(calib_dataset)` (or the CLI `hailo optimize` command), as described on the user guide on: Building Models / Model optimization / Model Optimization Workflow.
The calibration dataset should be preprocessed according to the model's input requirements and it is recommended to have at least 1024 inputs and to use a GPU.
During this step it is also possible to use a model script which change the default behavior of the Dataflow Compiler, for example, to add additional layer for normalization.
All the model script available commands are described in the user guide on: Building Models / Model optimization / Optimization Related Model Script Commands.

In order to learn how to deal with common pitfalls, image formats and accuracy, refer to the in-depth section.

In [34]:
# from ultralytics.utils.downloads import download
# # Download labels
# datadir = '../data'
# url = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/coco2017labels.zip'  # labels
# # download(url, dir=datadir)

# url_img = 'http://images.cocodataset.org/zips/val2017.zip'
# download(url_img, dir=datadir)

In [3]:
import torchvision as tv
import torch

def preproc(image, output_height=640, output_width=640):
    preprocess = tv.transforms.Compose([
        tv.transforms.Resize((output_height, output_width)),
    ])
    
    data = np.array(preprocess(image))
    
    return data

data_batch_size = 1500
images_path = "../data/visdrone/visdrone/VisDrone2019-DET-train/images" # use training data for calib
images_list = [img_name for img_name in os.listdir(images_path) if os.path.splitext(img_name)[1] == ".jpg"]
calib_dataset = np.zeros((data_batch_size, 640, 640, 3))
for idx, img_name in enumerate(sorted(images_list)):
    if idx==data_batch_size:
        break
    img = Image.open(os.path.join(images_path, img_name)).convert('RGB')
    img_preproc = preproc(img)
    calib_dataset[idx, :, :, :] = img_preproc

np.save("calib_set_visdrone.npy", calib_dataset)

In [36]:
calib_dataset.shape # should be (1500, 640, 640, 3)

(1500, 640, 640, 3)

In [25]:
# Second, we will load our parsed HAR from the Parsing Tutorial

model_name = "yolo11n_visdrone"
#yolo11n_hailo_model_visdroneop14.har
hailo_model_har_name = "yolo11n_visdrone_hailo_model_op14.har"
assert os.path.isfile(hailo_model_har_name), "Please provide valid path for HAR file"
runner = ClientRunner(har=hailo_model_har_name)
# By default it uses the hw_arch that is saved on the HAR. For overriding, use the hw_arch flag.

In [19]:

from pprint import pprint

try:
    # Access the HailoNet as an OrderedDict
    hn_dict = runner.get_hn()  # Or use runner._hn if get_hn() is unavailable
    print("Inspecting layers from HailoNet (OrderedDict):")

    # Pretty-print each layer
    for key, value in hn_dict.items():
        print(f"Key: {key}")
        pprint(value)
        print("\n" + "="*80 + "\n")  # Add a separator between layers for clarity

except Exception as e:
    print(f"Error while inspecting hn_dict: {e}")

Inspecting layers from HailoNet (OrderedDict):
Key: name
'yolo11n_visdrone'


Key: net_params
OrderedDict([('version', '1.0'),
             ('stage', 'HN'),
             ('clusters_placement', [[]]),
             ('clusters_to_skip', []),
             ('output_layers_order',
              ['yolo11n_visdrone/conv51',
               'yolo11n_visdrone/conv54',
               'yolo11n_visdrone/conv62',
               'yolo11n_visdrone/conv65',
               'yolo11n_visdrone/conv77',
               'yolo11n_visdrone/conv80']),
             ('is_transformer', True),
             ('transposed_net', False),
             ('net_scopes', ['yolo11n_visdrone']),
             ('lora_adapters', [])])


Key: layers
OrderedDict([('yolo11n_visdrone/input_layer1',
              OrderedDict([('type', 'input_layer'),
                           ('input', []),
                           ('output', ['yolo11n_visdrone/conv1']),
                           ('input_shapes', [[-1, 640, 640, 3]]),
               

In [32]:
# Now we will create a model script, that tells the compiler to add a normalization on the beginning
# of the model (that is why we didn't normalize the calibration set;
# Otherwise we would have to normalize it before using it)

# Batch size is 8 by default
alls =  """
normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])
change_output_activation(conv54, sigmoid)
change_output_activation(conv65, sigmoid)
change_output_activation(conv80, sigmoid)
nms_postprocess("../yolov11_nms_config_visdrone.json", meta_arch=yolov8, engine=cpu)
allocator_param(width_splitter_defuse=disabled)
 """
#model_optimization_flavor(optimization_level=4, compression_level=4) add this to alls for better opt


# Load the model script to ClientRunner so it will be considered on optimization
#runner.load_model_script(alls)
runner.load_model_script(alls)

# Call Optimize to perform the optimization process
runner.optimize(calib_dataset)

# Save the result state to a Quantized HAR file
quantized_model_har_path = f"{model_name}_quantized_model_visdrone.har"
runner.save_har(quantized_model_har_path)

[info] Loading model script commands to yolo11n_visdrone from string
[info] Starting Model Optimization
[info] Using default optimization level of 2
[info] Model received quantization params from the hn


ValueError: Exception encountered when calling layer "yolov8_nms_postprocess" (type HailoPostprocess).

in user code:

    File "/local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_model_optimization/acceleras/hailo_layers/base_hailo_none_nn_core_layer.py", line 45, in call  *
        outputs = self.call_core(inputs, training, **kwargs)
    File "/local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_model_optimization/acceleras/hailo_layers/hailo_postprocess.py", line 123, in call_core  *
        is_bbox_decoding_only=self.postprocess_type == PostprocessType.BBOX_DECODER,
    File "/local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_model_optimization/acceleras/hailo_layers/hailo_postprocess.py", line 157, in bbox_decoding_and_nms_call  *
        decoded_bboxes, detection_score = self.yolov8_decoding_call(inputs, offsets=[0.5, 0.5])
    File "/local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_model_optimization/accelera

In [ ]:
!hailomz eval yolov11n --har /local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_tutorials/notebooks/yolo11n_quantized_model_opl4.har --target=emulator

In [ ]:
!hailomz eval yolov11n --har /local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_tutorials/notebooks/yolo11n_quantized_model.har --target=emulator

In [ ]:
!hailomz eval yolov11n --har /local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_tutorials/notebooks/yolo11n_quantized_model.har

In [ ]:
!hailomz eval yolov11n

In [27]:
sample_dataset = np.zeros((2, 640, 640, 3))
SAMPLE_IMAGE_PATH = '../data/coco/images/val2017/000000000139.jpg'
img = Image.open(SAMPLE_IMAGE_PATH).convert('RGB')
img_preproc = preproc(img)
sample_dataset[0,:,:,:] = img_preproc

# #Notice that we use the original images, because normalization is IN the model
with runner.infer_context(InferenceContext.SDK_NATIVE) as ctx:
    modified_res = runner.infer(ctx, sample_dataset[:1, :, :, :])

[info] Using 1 GPU for inference


Inference: 0entries [04:50, ?entries/s]
Inference: 8entries [00:10,  1.32s/entries]


In [29]:
modified_res[0].shape

(1, 80, 80, 64)

In [ ]:

model_path = './yolo11n_quantized_model.har'
runner = ClientRunner(har=model_path)
with runner.infer_context(InferenceContext.SDK_QUANTIZED) as ctx:
    output = runner.infer(ctx, sample_dataset[:1, :, :, :])

In [ ]:
output[0].shape

In [ ]:
tv_dets = modified_res[:, 62, :, :].reshape(5,100) # this is classid=62, which is a television
tv_dets.transpose()[:2, :]

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.predict(sample_dataset[0,:,:,:], imgsz=640, conf=0.2)
# Process results list
for result in results:
    boxes = result.boxes  # Boxes object for bounding box outputs
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    #result.show()  # display to screen
    result.save(filename="result.jpg")  # save to disk
    
boxes[boxes.cls==62]

That concludes the quick tutorial.

In [ ]:
boxes[boxes.cls==62]

In [ ]:
import cv2
import numpy as np

# Load the image
SAMPLE_IMAGE_PATH = '../data/coco/images/val2017/000000000139.jpg'
img = cv2.imread(SAMPLE_IMAGE_PATH)

# Get image dimensions
height, width, _ = img.shape

# Ground truth labels in YOLO format (class_id, x_center, y_center, bbox_width, bbox_height)
labels = [
    (62, 0.39044347, 0.01051836, 0.6155355, 0.24089317),
    (62, 0.127641, 0.505153, 0.233312, 0.2227),
    (62, 0.934195, 0.583462, 0.127109, 0.184812)
]
#0.39044347, 0.01051836, 0.6155355 , 0.24089317, 0.9146729 ],
      # [0.48778272, 0.86963475, 0.6793633 , 0.999899  , 0.5476935 ]],
      #dtype=float32)

# Loop through labels and draw bounding boxes
for class_id, x_center, y_center, bbox_width, bbox_height in labels:
    # Convert normalized YOLO coordinates to pixel values
    x_center, y_center = int(x_center * width), int(y_center * height)
    bbox_width, bbox_height = int(bbox_width * width), int(bbox_height * height)

    # Calculate top-left and bottom-right corners
    x1 = int(x_center - bbox_width / 2)
    y1 = int(y_center - bbox_height / 2)
    x2 = int(x_center + bbox_width / 2)
    y2 = int(y_center + bbox_height / 2)

    # Draw the bounding box
    color = (0, 255, 0)  # Green color for bounding box
    thickness = 2
    cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

    # Add class label text
    cv2.putText(img, str(class_id), (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)



In [ ]:
# Display the image with bounding boxes
cv2.imshow("Ground Truth", img)
cv2.waitKey(0)  # Wait for key press
cv2.destroyAllWindows()  # Close window

In [ ]:
f =  open('../data/coco/labels/val2017/000000000139.txt', 'r')
for line in f:
    if line[:2] == '62':
        print(line)
